# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} content items")
df.head()

Loaded 30,000 content items


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df["signal_test"] = [1 if x == "down" else 0 for x in df["trend_direction"]]
df["signal_test"].head()

0    1
1    1
2    1
3    0
4    1
Name: signal_test, dtype: int64

In [3]:
df.groupby("freshness_tier")["signal_test"].agg(
    n="count",
    average="mean"
)

,n,average
freshness_tier,,
0-30,20480,0.511377
181+,174,0.471264
31-90,175,0.588571
91-180,9171,0.611057


### Signal check 1 — staleness and observed decline

I grouped pages by `freshness_tier` and measured the share with
`trend_direction == "down"` in each bucket. The table prints `n` so that
small buckets can be interpreted cautiously.

**Verdict: OPPOSITE.** The oldest `181+` bucket has an observed decline rate
of 47.1% (`n = 174`), below the 91–180-day bucket at 61.1% (`n = 9,171`).
In this snapshot, greater staleness does not correspond to a higher observed
decline rate. Therefore, I will not use staleness as a positive condition in
my baseline action score.

This comparison uses `trend_direction` only as an audit outcome; neither
`trend_direction` nor `trend_pct` will be used as an input to the rule.

In [4]:
position_data = df[df["avg_position"] > 0].copy()
position_data

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,signal_test
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.00,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.00,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.00,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.00,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.00,good,page_3_5,down,-34.7,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29994,content_995627a1f490,client_7f2253d7e2,70.0,0.05,LOW,0.00,keyword article,transactional,2801.0,18452.0,...,0.86,4.6,0.00,16.67,0.00,moderate,page_1,down,-65.8,1
29996,content_526572edb3fa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2654.0,17056.0,...,0.39,6.6,0.00,66.67,0.00,moderate,page_1,down,-75.1,1
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,0.19,4.1,0.00,0.00,0.00,good,page_1,down,-66.2,1
29998,content_ab26273a7e7a,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,0.22,6.0,1.73,4.06,0.00,excellent,page_1,down,-27.9,1


In [5]:
position_data["position_bucket"] = ["top_3" if x <= 3 else "page_1" if x <= 10 else "striking" if x <= 20 else "page_3_5" if x <= 50 else "deep" for x in position_data["avg_position"]]
position_data["position_bucket"].head()

0    striking
1    page_3_5
2    page_3_5
3      page_1
4    page_3_5
Name: position_bucket, dtype: object

In [6]:
position_data.groupby("position_bucket")["ctr"].agg(
    n="count",
    median_ctr="median"
)

,n,median_ctr
position_bucket,,
deep,1314,0.00
page_1,11842,0.16
page_3_5,7225,0.03
striking,7273,0.10
top_3,1141,0.00


### Signal check 2 — CTR versus position

I excluded `avg_position == 0` because zero represents missing position data.
I then grouped the remaining pages by position and calculated median CTR with
the number of pages in each bucket.

**Verdict: MIXED.** Median CTR is 0.16% for page-one positions (`n = 11,842`)
and falls to 0.10% in the striking-distance bucket (`n = 7,273`) and 0.03%
at positions 21–50 (`n = 7,225`). However, the top-three bucket has a median
CTR of 0.00%, showing that position alone does not explain CTR in this
snapshot. I will interpret CTR only together with sufficient impressions and
a realistic position range.

### Final baseline rule

Prioritize a page for a title, meta-description, and search-intent review when
it has at least 3,000 impressions, ranks between positions 4 and 20, and has
CTR at or below 0.5%. These pages have enough search visibility for a review
to matter, are close enough to page one for improvement to be plausible, and
have low CTR relative to their opportunity.

- **Action label:** `review_title_meta_and_intent`
- **Reason code:** `visible_low_ctr_striking_distance`
- **Excluded signal:** I do not score on staleness because the staleness audit
  produced an OPPOSITE verdict.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
queue = df.copy()

In [8]:
queue["visible"] = queue["impressions_90d"] >= 3000
queue["improvable_position"] = queue["avg_position"].between(4,20)
queue["low_ctr"] = queue["ctr"] <= 0.5
queue["eligible"] = queue["visible"] & queue["improvable_position"] & queue["low_ctr"]
queue[["visible", "improvable_position", "low_ctr", "eligible"]].head()

,visible,improvable_position,low_ctr,eligible
0,True,True,False,False
1,True,False,True,False
2,True,False,True,False
3,True,True,True,True
4,True,False,True,False


In [9]:
queue["baseline_score"] = (
    queue["impressions_90d"] * (21 - queue["avg_position"])
).where(queue["eligible"], 0)

In [10]:
queue.sort_values("baseline_score", ascending=False)[
    [
        "content_id",
        "impressions_90d",
        "avg_position",
        "ctr",
        "eligible",
        "baseline_score",
    ]
].head()

,content_id,impressions_90d,avg_position,ctr,eligible,baseline_score
6653,content_5fe46e04994d,517715,4.2,0.14,True,8697612.0
17812,content_aaef01a50def,517109,5.4,0.25,True,8066900.4
29879,content_1a9e894be2e2,416180,4.0,0.23,True,7075060.0
18870,content_db5989a78dd3,345111,5.4,0.21,True,5383731.6
26531,content_cb112fce36be,309910,5.6,0.16,True,4772614.0


In [11]:
queue["reason_code"] = ""
queue["action_label"] = ""

In [12]:
queue.loc[queue["eligible"], "reason_code"] = ("visible_low_ctr_striking_distance")
queue.loc[queue["eligible"], "action_label"] = ("review_title_meta_and_intent")
queue.loc[
    queue["eligible"],
    ["reason_code", "action_label"]
].head()

,reason_code,action_label
3,visible_low_ctr_striking_distance,review_title_meta_and_intent
5,visible_low_ctr_striking_distance,review_title_meta_and_intent
16,visible_low_ctr_striking_distance,review_title_meta_and_intent
17,visible_low_ctr_striking_distance,review_title_meta_and_intent
18,visible_low_ctr_striking_distance,review_title_meta_and_intent


In [13]:
review_columns = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "avg_position",
    "ctr",
]

ranked_queue = (
    queue.loc[queue["eligible"], review_columns]
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

In [14]:
output_path = Path("../outputs/baseline_action_score.csv")
output_path.parent.mkdir(exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print(f"Wrote {len(ranked_queue):,} rows to {output_path}")
ranked_queue.head(10)

Wrote 4,481 rows to ..\outputs\baseline_action_score.csv


,content_id,baseline_score,reason_code,action_label,impressions_90d,avg_position,ctr
0,content_5fe46e04994d,8697612.0,visible_low_ctr_striking_distance,review_title_meta_and_intent,517715,4.2,0.14
1,content_aaef01a50def,8066900.4,visible_low_ctr_striking_distance,review_title_meta_and_intent,517109,5.4,0.25
2,content_1a9e894be2e2,7075060.0,visible_low_ctr_striking_distance,review_title_meta_and_intent,416180,4.0,0.23
3,content_db5989a78dd3,5383731.6,visible_low_ctr_striking_distance,review_title_meta_and_intent,345111,5.4,0.21
4,content_cb112fce36be,4772614.0,visible_low_ctr_striking_distance,review_title_meta_and_intent,309910,5.6,0.16
5,content_36ff89c8214e,4042828.9,visible_low_ctr_striking_distance,review_title_meta_and_intent,295097,7.3,0.05
6,content_008fb02c46cb,3930929.8,visible_low_ctr_striking_distance,review_title_meta_and_intent,236803,4.4,0.26
7,content_aa4baf490b43,3869979.0,visible_low_ctr_striking_distance,review_title_meta_and_intent,256290,5.9,0.50
8,content_73c54f78c06a,3487596.9,visible_low_ctr_striking_distance,review_title_meta_and_intent,213963,4.7,0.10
9,content_c21024970297,3360719.4,visible_low_ctr_striking_distance,review_title_meta_and_intent,211366,5.1,0.41


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

| Content ID | Action | Why it is here | Confidence | What would make it wrong |
|---|---|---|---|---|
| `content_5fe46e04994d` | Review title, meta description, and intent | 517,715 impressions, position 4.2, CTR 0.14% | Medium | CTR may be driven by SERP features or a query mix that does not respond to snippet changes. |
| `content_aaef01a50def` | Review title, meta description, and intent | 517,109 impressions, position 5.4, CTR 0.25% | Medium | The page may already match search intent; low CTR could be normal for its result page. |
| `content_1a9e894be2e2` | Review title, meta description, and intent | 416,180 impressions, position 4.0, CTR 0.23% | Medium | The apparent opportunity may be mostly branded or navigational searches, where a refresh may not help. |
| `content_db5989a78dd3` | Review title, meta description, and intent | 345,111 impressions, position 5.4, CTR 0.21% | Medium | Search-result layout, ads, or rich results may attract clicks away from the ordinary result. |
| `content_cb112fce36be` | Review title, meta description, and intent | 309,910 impressions, position 5.6, CTR 0.16% | Medium | A title/meta revision may not solve a content relevance or competitor-trust issue. |
| `content_36ff89c8214e` | Review title, meta description, and intent | 295,097 impressions, position 7.3, CTR 0.05% | Medium | Position 7.3 may be too low for snippet changes alone to create meaningful gains. |
| `content_008fb02c46cb` | Review title, meta description, and intent | 236,803 impressions, position 4.4, CTR 0.26% | Medium | The page could be shown for broad, low-intent queries rather than a fixable snippet problem. |
| `content_aa4baf490b43` | Review title, meta description, and intent | 256,290 impressions, position 5.9, CTR 0.50% | Low | CTR is exactly at the cutoff, so this may be a threshold artefact rather than a clear weak pick. |
| `content_73c54f78c06a` | Review title, meta description, and intent | 213,963 impressions, position 4.7, CTR 0.10% | Medium | A zero/very-low CTR may reflect reporting or query-mix effects rather than page metadata. |
| `content_c21024970297` | Review title, meta description, and intent | 211,366 impressions, position 5.1, CTR 0.41% | Medium | The ranking and CTR may already be normal for its niche; human inspection could find no actionable change. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

`content_aa4baf490b43` is the clearest weak pick: its CTR is exactly 0.50%,
the rule cutoff. It may be included because of an arbitrary threshold rather
than a meaningful CTR problem.

`content_36ff89c8214e` is also uncertain. Its position is 7.3, so poor CTR
may be more a ranking problem than a title or meta-description problem.
These examples show why this is a decision-support queue for human review,
not proof that every recommended page should be changed.

### Leakage check

The baseline score uses only `impressions_90d`, `avg_position`, and `ctr`.
It does not use `trend_direction`, `trend_pct`, or the temporary
`signal_test` column; those were used only for the first signal audit.
It also does not use any last-30-day / previous-30-day trend fields or future
outcomes. Therefore, the ranked score is based only on decision-time snapshot
signals.

In [15]:
score_inputs = {"impressions_90d", "avg_position", "ctr"}

prohibited_inputs = {
    "trend_direction",
    "trend_pct",
    "signal_test",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
}

assert score_inputs.isdisjoint(prohibited_inputs)
print("Leakage check passed: score inputs contain no label-derived or trend-window fields.")

Leakage check passed: score inputs contain no label-derived or trend-window fields.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.